In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table
from src.evaluate import cross_validate_segments
from src.features import FEATURES, SIZE_FEATURES, SHAPE_FEATURES

feat = pd.read_csv(PROCESSED / "features_dev.csv")
table = load_fold_table("cv_folds_seizure.csv")
data = feat[feat["y"].isin([2, 3])].reset_index(drop=True)          # classes 2 and 3 from the development set
assert set(data["segment"]) <= set(table["segment"])

fold0 = data["segment"].map(table.set_index("segment")["repeat_0"])
c = pd.crosstab(fold0, data["y"])
print("segments:", len(data), data["y"].value_counts().sort_index().to_dict(), "| independent groups:", data["group"].nunique())
print("segments per fold:", sorted(c.sum(axis=1).unique()), "| class 2 per fold:", sorted(c[2].unique()), "| class 3 per fold:", sorted(c[3].unique()))

segments: 160 {2: 80, 3: 80} | independent groups: 76
segments per fold: [np.int64(15), np.int64(16), np.int64(17)] | class 2 per fold: [np.int64(8)] | class 3 per fold: [np.int64(7), np.int64(8), np.int64(9)]


In [2]:
feature_sets = {"all 23 features": FEATURES, "size only (8)": SIZE_FEATURES, "shape only (15)": SHAPE_FEATURES}
models = {
    "chance": lambda: DummyClassifier(strategy="prior"),
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
}
results = {}
for fname, cols in feature_sets.items():
    for mname, make_model in models.items():
        if mname == "chance" and fname != "all 23 features":
            continue
        results[(fname, mname)] = cross_validate_segments(make_model, data, cols, table, positive=2)

summary = pd.DataFrame({k: v.drop(columns="repeat").mean() for k, v in results.items()}).T
summary["auc_sd"] = pd.Series({k: v["auc_fold"].std() for k, v in results.items()})
summary[["auc_fold", "auc_sd", "pr_auc_fold", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy"]].round(3)

auc_fold  auc_sd  pr_auc_fold  \
all 23 features chance                  0.500   0.000        0.500   
                logistic regression     0.708   0.025        0.797   
                random forest           0.770   0.023        0.806   
size only (8)   logistic regression     0.664   0.012        0.748   
                random forest           0.738   0.025        0.780   
shape only (15) logistic regression     0.749   0.018        0.821   
                random forest           0.779   0.024        0.808   

                                     sensitivity  specificity  precision  \
all 23 features chance                     0.100        0.888      0.471   
                logistic regression        0.610        0.732      0.695   
                random forest              0.668        0.732      0.714   
size only (8)   logistic regression        0.560        0.688      0.642   
                random forest              0.610        0.753      0.711   
shape only (15) logistic regression        0.640        0.768      0.735   
                random forest              0.695        0.765      0.747   

                                        f1  balanced_accuracy  
all 23 features chance               0.165              0.494  
                logistic regression  0.650              0.671  
                random forest        0.690              0.700  
size only (8)   logistic regression  0.598              0.624  
                random forest        0.657              0.681  
shape only (15) logistic regression  0.684              0.704  
                random forest        0.720              0.730

In [3]:
group_class = data.groupby("group")["y"].agg(lambda s: s.value_counts().idxmax())
group_size = data.groupby("group").size()
n_pos = (data["y"] == 2).sum()
logistic = models["logistic regression"]

chance, s = [], 0
while len(chance) < 30:
    rng = np.random.default_rng(100 + s)
    s += 1
    shuffled = pd.Series(rng.permutation(group_class.values), index=group_class.index)    # labels shuffled between groups
    if abs(group_size[shuffled == 2].sum() - n_pos) > 4:
        continue
    d = data.copy()
    d["y"] = d["group"].map(shuffled)
    chance.append(cross_validate_segments(logistic, d, FEATURES, table, n_repeats=1, positive=2)["auc_fold"].iloc[0])
print("AUC with shuffled labels, 30 draws: 5th to 95th percentile", np.round(np.percentile(chance, [5, 95]), 2),
      "| mean", round(np.mean(chance), 3))

AUC with shuffled labels, 30 draws: 5th to 95th percentile [0.32 0.64] | mean 0.488


In [4]:
def null_range(make_model, cols, n_draws=30):
    chance, s = [], 0
    while len(chance) < n_draws:
        rng = np.random.default_rng(100 + s)
        s += 1
        shuffled = pd.Series(rng.permutation(group_class.values), index=group_class.index)
        if abs(group_size[shuffled == 2].sum() - n_pos) > 4:
            continue
        d = data.copy()
        d["y"] = d["group"].map(shuffled)
        chance.append(cross_validate_segments(make_model, d, cols, table, n_repeats=1, positive=2)["auc_fold"].iloc[0])
    return np.array(chance)

for mname in ["logistic regression", "random forest"]:
    for fname in ["all 23 features", "shape only (15)"]:
        c = null_range(models[mname], feature_sets[fname])
        print(f"{mname:20s} {fname:16s} shuffled-label 5th-95th: {np.round(np.percentile(c, [5, 95]), 2)} | mean {c.mean():.3f}")

logistic regression  all 23 features  shuffled-label 5th-95th: [0.32 0.64] | mean 0.488
logistic regression  shape only (15)  shuffled-label 5th-95th: [0.32 0.62] | mean 0.481
random forest        all 23 features  shuffled-label 5th-95th: [0.32 0.65] | mean 0.512
random forest        shape only (15)  shuffled-label 5th-95th: [0.35 0.66] | mean 0.499


In [5]:
from src.cv import make_group_fold_table
from src.data import load_raw, feature_cols
from src.groups import similarity_matrix, make_groups

raw = load_raw()
names_all, M_all = similarity_matrix(raw, feature_cols(raw))          # similarity between all 500 segments

settings = {"current (0.7 / 0.4)": (0.7, 0.4), "stricter (0.6 / 0.3)": (0.6, 0.3), "strictest (0.5 / 0.2)": (0.5, 0.2)}
use = {"all 23 features": FEATURES, "shape only (15)": SHAPE_FEATURES}
diag, auc = [], {}
for sname, (strong, weak) in settings.items():
    group_of = make_groups(names_all, M_all, strong=strong, weak=weak)
    d = data.assign(group=data["segment"].map(group_of).to_numpy())
    new_table = make_group_fold_table(d, group_of)
    size = d.groupby("group").size()
    f0 = d["segment"].map(new_table.set_index("segment")["repeat_0"])
    c = pd.crosstab(f0, d["y"])
    diag.append({"setting": sname, "groups": d["group"].nunique(), "biggest group": int(size.max()),
                 "groups with both classes": int((d.groupby("group")["y"].nunique() > 1).sum()),
                 "segments per fold": f"{c.sum(axis=1).min()}-{c.sum(axis=1).max()}"})
    for fname, cols in use.items():
        for mname in ["logistic regression", "random forest"]:
            auc[(fname, mname, sname)] = cross_validate_segments(models[mname], d, cols, new_table, positive=2)["auc_fold"].mean()

display(pd.DataFrame(diag))
display(pd.Series(auc).unstack().round(3))

,setting,groups,biggest group,groups with both classes,segments per fold
0,current (0.7 / 0.4),76,9,1,15-18
1,stricter (0.6 / 0.3),64,9,1,15-17
2,strictest (0.5 / 0.2),58,9,5,15-17


current (0.7 / 0.4)  \
all 23 features logistic regression                0.740   
                random forest                      0.762   
shape only (15) logistic regression                0.761   
                random forest                      0.768   

                                     stricter (0.6 / 0.3)  \
all 23 features logistic regression                 0.703   
                random forest                       0.739   
shape only (15) logistic regression                 0.737   
                random forest                       0.740   

                                     strictest (0.5 / 0.2)  
all 23 features logistic regression                  0.708  
                random forest                        0.739  
shape only (15) logistic regression                  0.742  
                random forest                        0.744

In [6]:
from scipy import signal as sps
from src.config import SAMPLING_RATE_HZ
from src.features import reconstruct_segments

raw_23 = raw[raw["segment"].isin(data["segment"])]
names_c, sigs = reconstruct_segments(raw_23, feature_cols(raw))

def control_features(x):
    xc = x - x.mean()
    f, p = sps.welch(xc, fs=SAMPLING_RATE_HZ, nperseg=512)
    df_hz = f[1] - f[0]
    band = lambda lo, hi: p[(f >= lo) & (f < hi)].sum() * df_hz
    total = p.sum() * df_hz
    return {"logpow_mains_45_55": np.log10(band(45, 55) + 1e-12), "logpow_noise_60_80": np.log10(band(60, 80) + 1e-12),
            "rel_mains": band(45, 55) / total, "rel_noise": band(60, 80) / total}

ctrl = pd.DataFrame([control_features(x) for x in sigs])
ctrl.insert(0, "segment", names_c)
data_c = data[["segment", "y", "group"]].merge(ctrl, on="segment")
ctrl_cols = ["logpow_mains_45_55", "logpow_noise_60_80", "rel_mains", "rel_noise"]

for mname in ["logistic regression", "random forest"]:
    r = cross_validate_segments(models[mname], data_c, ctrl_cols, table, positive=2)
    print(f"{mname:20s} negative-control AUC {r['auc_fold'].mean():.3f} (sd {r['auc_fold'].std():.3f})")

logistic regression  negative-control AUC 0.386 (sd 0.030)
random forest        negative-control AUC 0.528 (sd 0.040)


In [7]:
single = {}
for feature in FEATURES:
    r = cross_validate_segments(models["logistic regression"], data, [feature], table, positive=2)
    single[feature] = (r["auc_fold"].mean(), r["auc_fold"].std())
single = pd.DataFrame(single, index=["auc", "sd"]).T.sort_values("auc", ascending=False)
single["family"] = ["size" if f in SIZE_FEATURES else "shape" for f in single.index]
display(single.round(3))

,auc,sd,family
hjorth_complexity,0.697,0.010,shape
rel_alpha,0.641,0.024,shape
logratio_slow_fast,0.635,0.020,shape
skewness,0.621,0.012,shape
logpow_alpha,0.616,0.018,size
rel_delta,0.610,0.034,shape
logpow_theta,0.606,0.025,size
spectral_centroid,0.605,0.014,shape
logpow_beta,0.592,0.011,size
logratio_theta_alpha,0.580,0.022,shape


In [8]:
from sklearn.feature_selection import SelectKBest, f_classif

curve = {}
for k in (1, 2, 3, 5, 8, 12, 23):
    make_model = lambda k=k: make_pipeline(StandardScaler(), SelectKBest(f_classif, k=k), LogisticRegression(max_iter=1000))
    r = cross_validate_segments(make_model, data, FEATURES, table, positive=2)
    curve[k] = (r["auc_fold"].mean(), r["auc_fold"].std())
display(pd.DataFrame(curve, index=["auc", "sd"]).T.round(3))

,auc,sd
1,0.660,0.015
2,0.649,0.004
3,0.636,0.010
5,0.622,0.015
8,0.639,0.020
12,0.623,0.026
23,0.708,0.025


In [9]:
from scipy.signal import butter, sosfiltfilt
from src.features import segment_features

sos = butter(4, 40, btype="low", fs=SAMPLING_RATE_HZ, output="sos")          # smooth out everything above 40 Hz
filtered = np.stack([sosfiltfilt(sos, x) for x in sigs])                      # the 160 reconstructed segments
feats_f = pd.DataFrame([segment_features(x) for x in filtered])[FEATURES]
feats_f.insert(0, "segment", names_c)
data_f = data[["segment", "y", "group"]].merge(feats_f, on="segment")

orig = data.set_index("segment").loc[data_f["segment"]]
for f in ["hjorth_complexity", "hjorth_mobility", "line_length", "zero_crossing_rate", "spectral_centroid"]:
    print(f"{f:20s} median before {orig[f].median():8.3f} | after low-pass {data_f[f].median():8.3f}")

rows = {}
for fname, cols in {"all 23 features": FEATURES, "shape only (15)": SHAPE_FEATURES}.items():
    for mname in ["logistic regression", "random forest"]:
        rows[(fname, mname)] = cross_validate_segments(models[mname], data_f, cols, table, positive=2)["auc_fold"].mean()
rows[("hjorth_complexity alone", "logistic regression")] = cross_validate_segments(
    models["logistic regression"], data_f, ["hjorth_complexity"], table, positive=2)["auc_fold"].mean()
display(pd.Series(rows, name="AUC after 40 Hz low-pass").round(3))

hjorth_complexity    median before    3.840 | after low-pass    2.743
hjorth_mobility      median before    0.204 | after low-pass    0.188
line_length          median before    7.127 | after low-pass    6.610
zero_crossing_rate   median before    0.064 | after low-pass    0.058
spectral_centroid    median before    3.848 | after low-pass    3.832


all 23 features          logistic regression    0.717
                         random forest          0.764
shape only (15)          logistic regression    0.762
                         random forest          0.794
hjorth_complexity alone  logistic regression    0.651
Name: AUC after 40 Hz low-pass, dtype: float64

In [10]:
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier

family = {
    "logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVM (RBF)": lambda: make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE)),
    "random forest": lambda: RandomForestClassifier(200, random_state=RANDOM_STATE, n_jobs=-1),
    "gradient boosting": lambda: HistGradientBoostingClassifier(max_iter=100, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
}
results = {name: cross_validate_segments(mk, data, SHAPE_FEATURES, table, positive=2) for name, mk in family.items()}

auc = pd.DataFrame({name: r["auc_fold"] for name, r in results.items()})          # one row per repeat
diff = auc.sub(auc["logistic regression"], axis=0)                                 # difference from logistic, per repeat
summary = pd.DataFrame({"mean AUC": auc.mean(), "sd over repeats": auc.std(),
                        "mean diff vs logistic": diff.mean(), "repeats better than logistic (of 5)": (diff > 0).sum()}).round(3)
other = pd.DataFrame({name: r[["pr_auc_fold", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy"]].mean()
                      for name, r in results.items()}).T.round(3)
display(summary)
display(other)

/Users/bviveka/Projects/seizure-epileptogenic-localisation/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/bviveka/Projects/seizure-epileptogenic-localisation/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/bviveka/Projects/seizure-epileptogenic-localisation/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/bviveka/Project

,mean AUC,sd over repeats,mean diff vs logistic,repeats better than logistic (of 5)
logistic regression,0.749,0.018,0.000,0
SVM (RBF),0.793,0.015,0.044,5
random forest,0.779,0.024,0.029,5
gradient boosting,0.712,0.024,-0.038,1


,pr_auc_fold,sensitivity,specificity,precision,f1,balanced_accuracy
logistic regression,0.821,0.640,0.768,0.735,0.684,0.704
SVM (RBF),0.828,0.725,0.760,0.751,0.738,0.742
random forest,0.808,0.695,0.765,0.747,0.720,0.730
gradient boosting,0.754,0.668,0.658,0.661,0.664,0.662
